# 38 - Teleoperation And Data Factory

## What / Why / How

**What we are trying to do:** Design a teleoperation and data-collection loop for humanoid robot learning.

**Why this matters:** Human demonstrations are the practical fuel for current humanoid manipulation systems.

**How we will do it:** Simulate latency, action smoothing, episode metadata, data quality checks, and correction loops.

## Teleoperation Data Factory

Humanoid data is expensive. A practical builder needs teleop, metadata, quality checks, and correction loops.

In [ ]:
import math
import random
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

try:
    import matplotlib.pyplot as plt
    HAS_PLOT = True
except ModuleNotFoundError:
    plt = None
    HAS_PLOT = False

np.set_printoptions(precision=3, suppress=True)

def plot_unavailable():
    if not HAS_PLOT:
        print("Install matplotlib to see plots: pip install -r requirements.txt")

In [ ]:
rng = np.random.default_rng(38)
dt = 0.05
latency_steps = 3
human_commands = [np.array([0.04, 0.02]) * np.sin(i * 0.1) + np.array([0.03, 0.0]) for i in range(120)]
command_buffer = [np.zeros(2) for _ in range(latency_steps)]
robot_pos = np.zeros(2)
episode = []
for t, cmd in enumerate(human_commands):
    command_buffer.append(cmd)
    delayed = command_buffer.pop(0)
    smooth = 0.8 * delayed
    robot_pos = robot_pos + smooth + rng.normal(0, 0.002, 2)
    episode.append({"t": t * dt, "obs": robot_pos.copy(), "action": smooth.copy(), "latency_steps": latency_steps})
print("episode length:", len(episode))
print("last record:", episode[-1])

In [ ]:
actions = np.array([step["action"] for step in episode])
speeds = np.linalg.norm(actions, axis=1) / dt
quality = {
    "max_speed": float(speeds.max()),
    "mean_speed": float(speeds.mean()),
    "saturation_rate": float(np.mean(np.linalg.norm(actions, axis=1) > 0.045)),
    "missing_frames": 0,
}
print(quality)

In [ ]:
episode_card = {
    "task": "toy reach under teleoperation",
    "operator": "simulated",
    "robot": "2D end-effector proxy",
    "control_rate_hz": int(1 / dt),
    "latency_ms": latency_steps * dt * 1000,
    "quality": quality,
}
for k, v in episode_card.items():
    print(f"{k}: {v}")

In [ ]:
corrections = []
for i, step in enumerate(episode):
    if np.linalg.norm(step["action"]) > 0.045:
        corrections.append({"index": i, "reason": "too fast", "new_action": step["action"] * 0.7})
print("corrections needed:", len(corrections))
print(corrections[:3])

## Exercises

1. Add camera timestamps and detect action-observation misalignment.
2. Add success/failure labels.
3. Design a teleop setup for bimanual humanoid manipulation.